# 02. 전체 학습·추론 파이프라인 (Google Colab, A100 80GB)

최종 제출(LB 0.9357)에 쓰인 모든 모델 출력을 **원래 실행 순서대로** 다시 만드는 노트북입니다.
명령과 순서는 실행 로그와 작업 큐 기록에서 복원했습니다(`docs/pipeline_provenance.md`).

> 검증 범위
> - 이 노트북의 준비·x0·x1 셀은 Colab에서 실제로 실행해 확인했습니다.
> - GPU 단계는 원래 Colab 작업 큐로 실행된 명령을 그대로 옮긴 것입니다. 이 노트북 한 번으로 처음부터 끝까지 다시 돌려 보지는 않았습니다(약 15 GPU시간).
> - 제출 파일과의 정확한 일치는 `01_reproduce_best.ipynb`가 저장된 모델 출력으로 확인합니다.

| 항목 | 내용 |
|---|---|
| 하드웨어 | Colab A100 80GB ×1, High-RAM, 디스크 여유 150GB 이상 권장 |
| 소요 시간 | GPU 약 15시간(한 번에 한 작업 기준) + CPU 약 20분 |
| 결과 | `RUNS/`에 단계별 출력 → 마지막 셀에서 결합·할당량·하네스를 실행하고 제출 파일과 비교 |

꼭 지킬 점
1. **실험용 `KOALTQ_*` 환경변수는 명령마다만** 넘깁니다(아래 `run()`이 처리). 커널 전체에 설정하면 이후 단계가 바뀝니다.
2. **순서 제약**
   - x7·x15·x17 스태커는 판정기 출력 폴더(`x14_*`)가 정확히 13개일 때 실행합니다. 곧 rot1·rot6보다 먼저입니다.
   - 스태커가 runs 폴더의 판정기 결과를 모두 읽어 특징을 만들기 때문입니다.
   - x17fx는 rot1·rot6 이후(15개)에 실행합니다.
3. GPU 단계(LoRA 학습, VLM 추론)는 비트 단위 재현이 보장되지 않습니다.
   - 처음부터 다시 학습하면 확률이 조금 달라지고 최종 파일도 몇 행 다를 수 있습니다.
   - 제출 파일과의 정확한 일치는 `01_reproduce_best.ipynb`(저장된 출력 사용)로 확인합니다.
4. 최종 제출에 쓰인 LoRA 전체 학습본 어댑터는 보존되지 않았습니다. 이 노트북의 x6 단계가 같은 레시피로 다시 학습합니다.
5. Colab은 클라이언트 활동이 없으면 세션을 회수할 수 있으니 긴 단계 중에는 연결을 유지합니다.

## 0. 준비: 제출물 폴더 배치, 경로, 실행 도우미

In [ ]:
import os, subprocess, time, pathlib, json, glob, shutil

PKG = pathlib.Path("/content/KoAltQ_submission_0.9357")      # 제출물 폴더를 이 위치에 업로드/압축 해제
CODE, BUNDLE = PKG / "src", PKG / "data" / "koaltq_bundle"
USE_DRIVE = False                                            # True: 단계별 출력을 Google Drive에 보존 (세션이 끊겨도 이어서 실행)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
RUNS = pathlib.Path("/content/drive/MyDrive/koaltq_runs_rebuild" if USE_DRIVE else "/content/runs_rebuild")  # 단계별 출력 폴더
LOGS = RUNS / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
assert (CODE / "koaltq2" / "run.py").exists() and (BUNDLE / "data" / "boundary_test_rejudge.jsonl").exists()

BASE_ENV = {k: v for k, v in os.environ.items() if not k.startswith("KOALTQ_")}   # 남아 있는 실험 플래그 제거

def run(tag, args, env=None, force=False):
    """python -m koaltq2.run <args> 를 단계별 로그와 함께 실행 (env: 이 명령에만 적용할 KOALTQ_* 변수).
    같은 tag의 로그에 성공 기록이 있으면 건너뜀 (세션이 끊긴 뒤 다시 실행할 때 이어서 진행)."""
    lp = LOGS / f"{tag}.log"
    if not force and lp.exists() and " DONE in " in lp.read_text(encoding="utf-8", errors="ignore"):
        print(f"[{tag}] already done - skipped"); return
    e = dict(BASE_ENV, PYTHONPATH=str(CODE), KOALTQ_BUNDLE=str(BUNDLE), KOALTQ_RUNS=str(RUNS), **(env or {}))
    t0 = time.time()
    with open(LOGS / f"{tag}.log", "w") as f:
        p = subprocess.run(["python", "-u", "-m", "koaltq2.run", *args.split()], cwd="/content", env=e,
                           stdout=f, stderr=subprocess.STDOUT)
    print(f"[{tag}] rc={p.returncode} {time.time() - t0:.0f}s  ({args}{' | ' + str(env) if env else ''})")
    if p.returncode != 0:
        raise RuntimeError(open(LOGS / f"{tag}.log", encoding="utf-8", errors="ignore").read()[-3000:])

def n_dirs(pattern):
    return len([d for d in glob.glob(str(RUNS / pattern)) if os.path.isdir(d)])

## 1. 패키지 설치 + 환경 확인 (x0)

검증한 런타임: Python 3.13.16, torch 2.11.0+cu130, NVIDIA driver 580.82.07, CUDA 13.0 (`requirements-gpu.txt`).
Colab 기본 torchao는 PEFT 0.21.2와 맞지 않아 제거합니다.

In [ ]:
# torch 2.11.0 + CUDA 13.0 휠 (Colab 기본 이미지가 다르면 실행): 
# !pip install -q torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu130
!pip install -q 'transformers==5.22.1' 'peft==0.21.2' 'bitsandbytes==0.50.2' 'accelerate==1.15.0' 'flash-linear-attention==0.6.0' 'lightgbm==4.6.0'
!pip uninstall -y -q torchao
!python -c "import torch, transformers, peft; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| cuda', torch.cuda.is_available())"
run("x0", "x0", {"KOALTQ_CONTENT": str(PKG / "data")})   # 번들 MANIFEST(3,007개 sha256) + 데이터 계약 검사 (2400/600행, fold sha, 경계사례 109행)

## 2. 기초 특징 (약 1시간)

| 단계 | 내용 | 시간(원래) |
|---|---|---|
| x1 | 규칙·구조 특징 + TF-IDF → GBM/LR (CPU) | 2분 |
| x2 | PaddleOCR PP-OCRv5 korean (격리 venv 자동 생성) → OCR 특징 | 13분 |
| x3 | 고정 VLM 임베딩: Qwen3.5-9B, InternVL3.5-14B → LR/MLP | 21분, 41분 |

In [ ]:
run("x1", "x1")
run("x2", "x2 paddle")
run("x3_qwen35_9b", "x3 --model qwen35_9b")
run("x3_internvl35_14b", "x3 --model internvl35_14b")
shutil.rmtree("/root/.cache/huggingface/hub/models--OpenGVLab--InternVL3_5-14B-HF", ignore_errors=True)   # 디스크 확보 (원래 실행과 동일)

## 3. 9B LoRA, 교차 인코더, VLM 속성 질의 (약 5.5시간)

| 단계 | 내용 | 시간(원래) |
|---|---|---|
| x6 9B fold | Qwen3.5-9B LoRA, fold 0 검증 | 85분 |
| x11 | klue/roberta-large 교차 인코더 | 21분 |
| x12 ×3 | 예/아니오 속성 질의: Qwen3.5-9B, Qwen3-VL-8B, Qwen3-VL-32B | 48분, 39분, 46분 |
| x6 9B full | 학습 2,400 + 공개 109건 전체 학습 → test 확률 (최종 결합에 사용) | 73분 |

In [ ]:
run("x6_qwen35_9b_fold", "x6 --mode fold --backbone qwen35_9b")
run("x11", "x11")
run("x12_qwen35_9b", "x12 --model qwen35_9b")
run("x6_qwen35_9b_full", "x6 --mode full --backbone qwen35_9b")
run("x12_qwen3vl_8b", "x12 --model qwen3vl_8b")
run("x12_qwen3vl_32b", "x12 --model qwen3vl_32b")

## 4. 가이드라인 판정기 (약 5시간)

라벨링 가이드라인 전문과 레코드(이미지·alt·문맥·OCR)를 주고 선택지 글자의 확률을 읽습니다.
- 위치 편향을 줄이려고 선택지 순서를 바꾼 변형(rev, rot2~rot5)과 문맥·해상도 변형을 함께 만듭니다.
- 공통 지시문은 KV 캐시로 재사용합니다.

`x14_qwen38_27b_long`은 원래 두 번에 나눠 실행됐습니다.
- 앞 800행: 캐시 검사 허용치 0.03에서 캐시 없이
- 나머지: 허용치 0.08에서 캐시 사용

아래도 같은 방식으로 재현합니다.

In [ ]:
run("x14_qwen3vl_32b", "x14 --model qwen3vl_32b")
run("x14_qwen3vl_32b_long", "x14 --model qwen3vl_32b --guide long")
run("x14_qwen38_27b", "x14 --model qwen38_27b")
run("x14_qwen3vl_8b", "x14 --model qwen3vl_8b")
run("x14_qwen35_9b", "x14 --model qwen35_9b")
run("x14_qwen38_27b_long_a", "x14 --model qwen38_27b --guide long --limit 800")
run("x14_qwen38_27b_long_b", "x14 --model qwen38_27b --guide long", {"KOALTQ_PREFIX_TOL": "0.08"})
run("x12_qwen38_27b", "x12 --model qwen38_27b --questions q_decor,q_photo,q_info")
T8 = {"KOALTQ_PREFIX_TOL": "0.08"}
run("x14_hires", "x14 --model qwen38_27b --guide long --tag hires", dict(T8, KOALTQ_MAX_IMAGE_TOKENS="2048"))
run("x14_revctx", "x14 --model qwen38_27b --guide long --order rev --tag revctx", dict(T8, KOALTQ_CTX_CHARS="800", KOALTQ_OCR_CHARS="800"))
run("x16_insuff", "x14 --model qwen38_27b --guide long --ask insuff", T8)
run("x16_ok", "x14 --model qwen38_27b --guide long --ask ok", T8)
run("x14_ctx", "x14 --model qwen38_27b --guide long --tag ctx", dict(T8, KOALTQ_CTX_CHARS="800", KOALTQ_OCR_CHARS="800"))
for k in (2, 4, 3, 5):                                   # 원래 실행 순서
    run(f"x14_rot{k}", f"x14 --model qwen38_27b --guide long --order rot{k} --tag rot{k}", T8)

## 5. 체크포인트 → 스태커 (CPU, 약 8분)

스태커는 `x3_*/feats.npy`, `x12_*`, `x14_*`, `x16_*`를 모두 읽어 특징을 만듭니다.
그래서 이 시점에 `x14_*`가 **정확히 13개**여야 원래와 같은 특징(x7j·x17 364개, x15 124개)이 나옵니다.

In [ ]:
assert n_dirs("x14_*") == 13 and n_dirs("x16_*") == 2 and n_dirs("x12_*") == 4, (n_dirs("x14_*"), n_dirs("x16_*"), n_dirs("x12_*"))
run("x7", "x7")
run("x15", "x15")
run("x17", "x17")          # 학습 2,400 + 공개 109건 (데이터셋_라벨)
nf = {k: json.load(open(RUNS / k / "metrics.json"))["n_features"] for k in ("x7j_stack_lgbm", "x15_judge_lr", "x17_stack_lgbm_b109")}
print(nf)
assert nf == {"x7j_stack_lgbm": 364, "x15_judge_lr": 124, "x17_stack_lgbm_b109": 364}, "특징 수가 원래 실행과 다름 (x14_* 폴더 구성 확인)"

## 6. 27B LoRA와 나머지 판정기 순서 변형 (약 3.5시간)

| 단계 | 내용 | 시간(원래) |
|---|---|---|
| x6 27B fold | Qwen3.8-27B LoRA, fold 0 검증 (accum 16, 1.5 epoch) | 85분 |
| x14 rot1, rot6 | 판정기 순서 변형 2개 | 각 15분 |
| x6 27B full | 학습 2,400 + 공개 109건 전체 학습 → test 확률 (최종 결합에 사용) | 약 105분 |

In [ ]:
run("x6_qwen38_27b_fold", "x6 --mode fold --backbone qwen38_27b --accum 16 --epochs 1.5")
for k in (1, 6):
    run(f"x14_rot{k}", f"x14 --model qwen38_27b --guide long --order rot{k} --tag rot{k}", T8)
run("x6_qwen38_27b_full", "x6 --mode full --backbone qwen38_27b --accum 16 --epochs 1.5")

## 7. x17fx: 규칙 확정 test 행까지 학습한 스태커 (CPU, 약 2분)

- 학습 데이터: 학습 2,400 + 공개 109 + 규칙 확정 test 행
  - R2 복원 역할 표시 116행
  - R4′ 템플릿 쌍둥이 17행: 경계사례 재판정_라벨 전파로 정해진 행. Q&A 678상 사용 가능
- 이 시점에는 `x14_*`가 15개(378개 특징)입니다.
- 최종 결합은 이 모델의 **test 확률**만 씁니다.

In [ ]:
assert n_dirs("x14_*") == 15
run("x17fx", "x17", {"KOALTQ_X17_EXTRA": "1"})
assert json.load(open(RUNS / "x17_stack_lgbm_b109fx" / "metrics.json"))["n_features"] == 378


## 8. 결정 단계: 결합 → 할당량 → 하네스, 그리고 제출 파일과 비교

`scripts/reproduce_best.sh`와 같은 조립을 하고 x8을 실행합니다.
원래 실행에서 x8이 고른 결합 규칙은 `blend:x17_stack_lgbm_b109`(판정기 LR 비중 0.1)였습니다.
다시 학습한 출력에서는 후보 점수가 근소하게 바뀌어 다른 규칙이 뽑힐 수 있으니 확인합니다.

In [ ]:
W = pathlib.Path("/content/runs_rebuild_x8")
shutil.rmtree(W, ignore_errors=True); shutil.copytree(RUNS, W)
shutil.copy(W / "x17_stack_lgbm_b109fx" / "test.npy", W / "x17_stack_lgbm_b109" / "test.npy")
for d in ("x17_stack_lgbm_b109fx", "x6_qwen38_27b_fx_lora_full", "x6_qwen38_27b_fx3_lora_full"):
    shutil.rmtree(W / d, ignore_errors=True)
e = dict(BASE_ENV, PYTHONPATH=str(CODE), KOALTQ_BUNDLE=str(BUNDLE), KOALTQ_RUNS=str(W), KOALTQ_HARNESS="1")
p = subprocess.run(["python", "-m", "koaltq2.run", "x8", "--objective", "lbq", "--wlora", "0.4"], cwd="/content", env=e, capture_output=True, text=True)
print("\n".join(l for l in (p.stdout + p.stderr).splitlines() if "rule" in l or "[x8] harness" in l or "[submission]" in l)[-3000:])

import csv, collections, hashlib
rd = lambda q: {r["record_id"]: r["label"] for r in csv.DictReader(open(q, encoding="utf-8-sig"))}
new = W / "submissions" / "S3_ensemble_quota_l0_lbq.csv"
sub = rd(PKG / "submission" / "final_submission_0.9357.csv"); out = rd(new)
diff = collections.Counter(f"{sub[k]}->{out[k]}" for k in sub if sub[k] != out[k])
print("rows different from the submitted file:", sum(diff.values()), dict(diff))
print("sha256:", hashlib.sha256(open(new, "rb").read()).hexdigest(), "(submitted: 87c3de688bd20937...)")

### 재학습 결과를 볼 때의 기준값

- OCR `ocr.jsonl` sha256 `3cb586c8…`
- x3 cards_hash `7f50b370a8578720`
- x6 train_cards_hash `a553c8ba144af259`(fold), `3a2298a5e582e082`(full)
- 특징 수: x7j·x17 364, x15 124, x17fx 378
- x8 결합 규칙 `blend:x17_stack_lgbm_b109`(a = 0.1)
- 저장된 출력으로 만든 최종 sha256 `87c3de688bd20937…`